# INV-M365-V — Service Provisioning Site Detection Alignment

## Purpose

Prove the deterministic fix for `provision_service` when an existing SMARTHAUS group-connected site does not live at `/sites/{mail_nickname}`.

## Lemma Mapping

- Supports `L20`
- Supports `plan:m365-enterprise-readiness-master-plan:C1C`

## Invariant Support

- `INV-M365-V`

## Expected Results

- existing service groups resolve through the group root-site relationship
- the runtime no longer depends on guessed `/sites/{mail_nickname}` paths for existing services
- the live HR service surface is represented by `mail_nickname=hr` and `site_url=https://smarthausgroup.sharepoint.com/sites/hr2`


## Proof Summary

The live HR service surface showed that `mail_nickname=hr` resolves to a valid Microsoft 365 group while the connected SharePoint site lives at `/sites/hr2`, not `/sites/hr`. Deterministic service provisioning therefore has to resolve the group's actual root site first and only use path lookup as a bounded fallback.


In [ ]:
live_hr_surface = {
    "mail_nickname": "hr",
    "group_id": "e5634ed7-bdb4-46ef-884e-4d615c58cbe7",
    "site_id": "smarthausgroup.sharepoint.com,3e3fca26-bce9-4ee2-b0c9-ff3d28c3624c,2a6d0c31-a6d6-465d-b495-0c3ec1fa758b",
    "site_url": "https://smarthausgroup.sharepoint.com/sites/hr2",
}

def resolve_service_site(group_root_site: dict | None, path_site: dict | None) -> dict | None:
    if group_root_site and group_root_site.get("id"):
        return group_root_site
    if path_site and path_site.get("id"):
        return path_site
    return None

root_site = {"id": live_hr_surface["site_id"], "webUrl": live_hr_surface["site_url"]}
guessed_path_site = None
resolved = resolve_service_site(root_site, guessed_path_site)

assert resolved is not None
assert resolved["id"] == live_hr_surface["site_id"]
assert resolved["webUrl"] == live_hr_surface["site_url"]
